In [33]:
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Display t-SNE for Baseline

In [38]:
from data.swarmset import SwarmDataset
from src.eval.tsne import embed_tsne
# Compute Class confusion for the behavior vector
import numpy as np
%matplotlib

# TSNE for baseline
class_labels = np.loadtxt("~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/data/oracle_labels/HeRoVal.csv", dtype=int, delimiter=",")
val_ds = SwarmDataset("~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/data/HeRo-Val", rank=0, video=True)

labels = []
X = []
for i, (_id, _class) in enumerate(class_labels):
    labels.append(_class)
    X.append(val_ds[_id][2])

embed_tsne(np.abs(np.array(X)), labels)

Using matplotlib backend: MacOSX


# Display t-SNE for SimCLR

In [54]:
from data.swarmset import SwarmDataset
from src.eval.tsne import embed_tsne
# Compute Class confusion for the behavior vector
import numpy as np
%matplotlib

# TSNE for baseline
class_labels = np.loadtxt("~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/data/oracle_labels/HeRoVal.csv", dtype=int, delimiter=",")
val_ds = SwarmDataset("~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/data/HeRo-Val", rank=0, video=True)

from src.networks.simclr import SimCLR
model = SimCLR.load("~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/checkpoints/embeddings/seed1.pt")
model.eval()

labels = []
X = []
for i, (_id, _class) in enumerate(class_labels):
    labels.append(_class)
    vec = model.encode(val_ds[_id][0][::3]).detach().cpu()
    vec = vec.flatten()
    print(vec.shape)
    X.append(vec.tolist())

embed_tsne(np.abs(np.array(X)), labels)

Using matplotlib backend: MacOSX
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])


~/anaconda3/envs/BehaviorDiscovery/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
~/anaconda3/envs/BehaviorDiscovery/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size([512])
torch.Size

# Class Confusion: Baseline

In [39]:
# TSNE for baseline
class_labels = np.loadtxt("~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/data/oracle_labels/HeRoVal.csv", dtype=int, delimiter=",")
val_ds = SwarmDataset("~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/data/HeRo-Val", rank=0, video=True)

labels = []
X = []
for i, (_id, _class) in enumerate(class_labels):
    labels.append(_class)
    X.append(val_ds[_id][2])
    
# Class Confusion Matrix
X = np.array(X)
n = max(labels) + 1
confm = np.zeros((n, n))
print(confm)

# The number 1 winner of the ugliest O(n^3) loop in history.
c_set = list(zip(X, labels))
for l in range(len(c_set)):
    if l % 1e4 == 0:
        print("Iteration {}".format(l))
    x, _classX = c_set[l]
    for j in range(l, len(c_set)):
        y, _classY = c_set[j]
        if np.equal(x, y).all() or _classY != _classX:
            continue
        for k in range(len(c_set)):
            z, _classZ = c_set[k]
            # if _classZ == 0:
            #     continue
            if _classZ == _classX or np.equal(x, z).all() or np.equal(y, z).all():
                continue
            if np.linalg.norm(x - y) < np.linalg.norm(x - z):
                confm[_classX][_classY] += 1
            else:
                confm[_classX][_classZ] += 1
print(confm)

[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Iteration 0
[[6156334.  212590.  870468. 1213234.]
 [ 114110.  242629.   63554.   30157.]
 [ 433261.   91739. 1973613.  138333.]
 [ 228547.    9992.   47181.  567741.]]


In [40]:
for i in range(len(confm)):
    confm[i, :] /= np.sum(confm[i, :])
confm = np.round(confm, 3).T
confm.tolist()

[[0.728, 0.253, 0.164, 0.268],
 [0.025, 0.539, 0.035, 0.012],
 [0.103, 0.141, 0.748, 0.055],
 [0.144, 0.067, 0.052, 0.665]]

# Class Confusion: SimCLR

In [45]:
from src.networks.simclr import SimCLR
model = SimCLR.load("~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/checkpoints/embeddings/seed3.pt")
model.eval()

class_labels = np.loadtxt("~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/data/oracle_labels/HeRoVal.csv", dtype=int, delimiter=",")
val_ds = SwarmDataset("~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/data/HeRo-Val", rank=0, video=True)

labels = []
X = []
for i, (_id, _class) in enumerate(class_labels):
    labels.append(_class)
    print(val_ds[_id][0].shape)
    X.append(model.encode(val_ds[_id][0][::3]).detach().cpu().numpy())

print("Forward Passes Complete")
    
# Class Confusion Matrix
X = np.array(X)
n = max(labels) + 1
confm = np.zeros((n, n))
print(confm)

# The number 1 winner of the ugliest O(n^3) loop in history.
c_set = list(zip(X, labels))
for l in range(len(c_set)):
    if l % 1e4 == 0:
        print("Iteration {}".format(l))
    x, _classX = c_set[l]
    for j in range(l, len(c_set)):
        y, _classY = c_set[j]
        if np.equal(x, y).all() or _classY != _classX:
            continue
        for k in range(len(c_set)):
            z, _classZ = c_set[k]
            # if _classZ == 0:
            #     continue
            if _classZ == _classX or np.equal(x, z).all() or np.equal(y, z).all():
                continue
            if np.linalg.norm(x - y) < np.linalg.norm(x - z):
                confm[_classX][_classY] += 1
            else:
                confm[_classX][_classZ] += 1
print(confm)

(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 64, 64)
(9, 

In [46]:
for i in range(len(confm)):
    confm[i, :] /= np.sum(confm[i, :])
confm = np.round(confm, 3).T
confm.tolist()

[[0.64, 0.155, 0.22, 0.198],
 [0.067, 0.698, 0.054, 0.032],
 [0.173, 0.105, 0.678, 0.067],
 [0.12, 0.043, 0.048, 0.703]]